# 03 · Estoque da Vulto: raio-x

Fontes: `banco_estoque.csv` (saldo atual), `banco_entradas_estoque.csv` (entradas e transferências) e `contagem_set26.csv` (contagem física de set/26, as 5 abas de `contagem_inventario_set26.xlsx`).

Objetivo: **entender** o estoque, sem decidir nem corrigir nada. As mesmas 4 perguntas.

**Como é o Excel da contagem** (só olhar; nunca salvar pelo Excel): uma aba por unidade, cada uma num formato. Pinheiros tem título e data (20/09) em cima; Eldorado tem a coluna do conferente; **Savassi foi contada por modelo**, não por SKU; Batel tem observações como "faltou contar provador" e quantidades como "2 (1 no estoque + 1 vitrine)"; o CD é a posição do WMS de 28/09, com saldo, reservado e disponível.

> Antes de commitar: *Kernel → Restart & Clear Output*.

In [ ]:
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

In [ ]:
estoque = pd.read_csv(STAGING / "banco_estoque.csv", dtype=str)
entradas = pd.read_csv(STAGING / "banco_entradas_estoque.csv", dtype=str)
contagem = pd.read_csv(STAGING / "contagem_set26.csv", dtype=str)
lojas = pd.read_csv(STAGING / "banco_lojas.csv", dtype=str)
for tabela in (estoque, entradas):
    tabela["q"] = pd.to_numeric(tabela["qtd"])
contagem["q"] = pd.to_numeric(contagem["quantidade"])

## 1. O que é cada linha?

- `estoque`: o saldo **atual** de um SKU numa loja (a foto de hoje; não guarda o passado).
- `entradas`: uma entrada de mercadoria ou transferência.
- `contagem`: um item contado fisicamente numa unidade.

In [ ]:
print("estoque:", estoque.shape, "| SKU + loja nunca se repete:", not estoque.duplicated(["sku", "loja_id"]).any())
print("entradas:", entradas.shape, "| contagem:", contagem.shape)
display(estoque.head(3), entradas.head(3), contagem.head(3))

## 2. O que significa cada coluna?

In [ ]:
# saldo atual por loja (1–4 lojas, 5 site, 6 app, 7 CD)
estoque.groupby("loja_id")["q"].agg(["count", "sum", "min", "max"])

In [ ]:
# entradas por loja; no CD, a maioria das linhas não tem quantidade
display(entradas.groupby("loja_id")["q"].agg(["count", "sum"]).assign(sem_quantidade=entradas.groupby("loja_id")["q"].apply(lambda s: s.isna().sum())))
print("custo vazio:", entradas["custo_unit"].isna().sum(), "de", len(entradas))
print("fornecedor:", entradas["fornecedor"].value_counts().head(10).to_dict())
entradas["obs"].value_counts().head(12)

In [ ]:
# contagem: data, nível (SKU ou modelo) e reservado por unidade
contagem.groupby("unidade").agg(linhas=("codigo", "size"), data=("data_contagem", lambda s: ", ".join(s.dropna().unique()) or "(sem data)"),
                                nivel=("nivel", lambda s: ", ".join(s.unique())), com_reservado=("reservado", lambda s: s.notna().sum()))

## 3. Que problemas aparecem?

In [ ]:
print("saldos negativos:", (estoque["q"] < 0).sum(), "| zerados:", (estoque["q"] == 0).sum())
estoque.sort_values("q").head(5)

In [ ]:
import re

# Teste: o mesmo SKU escrito de 3 jeitos (VLT-CAM-004-BRA-M, CAM004BM, vlt cam 004 branco m) vira a mesma chave
# CAT + NNN + inicial da cor + tamanho. Serve só para medir; a regra de verdade é decisão da etapa 4.
def chave(sku):
    s = str(sku).upper().strip()
    longo = re.fullmatch(r"VLT[- ]([A-Z]{3})[- ](\d{3})[- ]([A-Z]+)[- ]?(.*)", s)
    if longo:
        return f"{longo[1]}{longo[2]}{longo[3][0]}{longo[4].replace(' ', '').replace('-', '')}"
    curto = re.fullmatch(r"([A-Z]{3})(\d{3})([A-Z])(.+)", s)
    if curto:
        return f"{curto[1]}{curto[2]}{curto[3]}{curto[4]}"
    return None

In [ ]:
estoque["chave"] = estoque["sku"].map(chave)

# a mesma peça com dois códigos na mesma loja (ex.: longo com saldo positivo, curto com saldo negativo)
codigos_por_peca = estoque.groupby(["loja_id", "chave"])["sku"].nunique()
print("mesma peça com 2 códigos na mesma loja:", (codigos_por_peca > 1).sum())

# site (5) e app (6): a mesma peça com a mesma quantidade nas duas "lojas"?
por_loja = estoque.pivot_table(index="chave", columns="loja_id", values="q", aggfunc="sum")
nas_duas = por_loja[["5", "6"]].dropna()
print("peças no site e no app:", len(nas_duas), "| com a mesma quantidade:", (nas_duas["5"] == nas_duas["6"]).sum())

In [ ]:
# contagem física × sistema, por unidade
unidade_loja = {"Pinheiros": "1", "Eldorado": "2", "Savassi": "3", "Batel": "4", "CD Cajamar": "7"}
pd.DataFrame([{"unidade": u, "contado": contagem.loc[contagem["unidade"] == u, "q"].sum(),
               "sistema": estoque.loc[estoque["loja_id"] == l, "q"].sum(),
               "sistema (só positivos)": estoque.loc[(estoque["loja_id"] == l) & (estoque["q"] > 0), "q"].sum()}
              for u, l in unidade_loja.items()])

## 4. Como se liga às outras fontes?

O SKU liga o estoque às vendas e ao catálogo; a contagem liga pelo `codigo`. Um teste importante: dá para reconstruir o estoque a partir das entradas e das vendas?

In [ ]:
catalogo = pd.read_csv(STAGING / "catalogo_produtos.csv", dtype=str)
print("SKU do estoque no catálogo: escrito igual", f"{estoque['sku'].isin(catalogo['sku']).mean():.0%}",
      "| pela chave", f"{estoque['chave'].isin(set(catalogo['sku'].map(chave).dropna())).mean():.0%}")
pecas_contadas = contagem[contagem["nivel"] == "sku"]
print("códigos contados (por SKU) que existem no estoque, pela chave:",
      f"{pecas_contadas['codigo'].map(chave).isin(set(estoque['chave'].dropna())).mean():.0%}")

In [ ]:
# teste: estoque reconstruído (entradas - vendas desde jan/25) × contagem física
vendas = pd.read_csv(STAGING / "banco_vendas.csv", dtype=str)
itens = pd.read_csv(STAGING / "banco_itens_venda.csv", dtype=str).merge(vendas[["id", "loja_id", "status", "criado_em"]], left_on="venda_id", right_on="id")
itens = itens[~itens["status"].fillna("").str.lower().isin(["cancelado", "cancelada"])]
itens["q"], itens["chave"] = pd.to_numeric(itens["qtd"]), itens["sku"].map(chave)
entradas["chave"] = entradas["sku"].map(chave)

linhas = []
for unidade, loja, ate in [("Pinheiros", "1", "2026-09-21"), ("Eldorado", "2", "2026-10-01"), ("Batel", "4", "2026-10-01")]:
    entrou = entradas[(entradas["loja_id"] == loja) & (entradas["data"] < ate)].groupby("chave")["q"].sum()
    saiu = itens[(itens["loja_id"] == loja) & (itens["criado_em"] < ate)].groupby("chave")["q"].sum()
    contado = contagem[contagem["unidade"] == unidade].assign(chave=lambda d: d["codigo"].map(chave)).groupby("chave")["q"].sum()
    reconstruido = entrou.sub(saiu, fill_value=0)
    comparacao = pd.DataFrame({"contado": contado, "reconstruido": reconstruido}).fillna(0)
    linhas.append({"unidade": unidade, "contado": comparacao["contado"].sum(), "reconstruído": comparacao["reconstruido"].sum(),
                   "peças que batem": f"{(comparacao['contado'] == comparacao['reconstruido']).mean():.0%}"})
pd.DataFrame(linhas)

## Desafios da revisão

In [ ]:
# o espelho site/app é o estoque do CD copiado?
wms_cd = contagem[contagem["unidade"] == "CD Cajamar"].assign(chave=lambda d: d["codigo"].map(chave)).groupby("chave")["q"].sum()
espelhadas = nas_duas[nas_duas["5"] == nas_duas["6"]].join(wms_cd.rename("wms_cd"), how="inner")
print("peças espelhadas com posição no WMS:", len(espelhadas), "| quantidade do site = saldo do WMS do CD:", (espelhadas["5"] == espelhadas["wms_cd"]).sum())
site_app = estoque["loja_id"].isin(["5", "6"])
print("linhas do site/app atualizadas em 29/09/2026:", (site_app & estoque["atualizado_em"].str.startswith("2026-09-29")).sum(), "de", site_app.sum())

In [ ]:
# a contagem dá mais que o sistema em todas as peças?
linhas = []
for unidade, loja in [("Pinheiros", "1"), ("Eldorado", "2"), ("Batel", "4")]:
    contado = contagem[contagem["unidade"] == unidade].assign(chave=lambda d: d["codigo"].map(chave)).groupby("chave")["q"].sum()
    sistema = estoque[estoque["loja_id"] == loja].groupby("chave")["q"].sum()
    j = pd.DataFrame({"contado": contado, "sistema": sistema}).fillna(0)
    linhas.append({"unidade": unidade, "contado > sistema": (j["contado"] > j["sistema"]).sum(), "igual": (j["contado"] == j["sistema"]).sum(),
                   "contado < sistema": (j["contado"] < j["sistema"]).sum(), "só no sistema": ((j["contado"] == 0) & (j["sistema"] > 0)).sum()})
pd.DataFrame(linhas)

## Anotações para o dicionário de dados

**O que é cada linha**
- `estoque` (7.130): o saldo **atual** de um SKU numa loja; SKU + loja nunca se repete. É só a foto de hoje: um ajuste sobrescreve o saldo, sem guardar o anterior.
- `entradas` (17.331): uma entrada de mercadoria (fornecedor) ou uma transferência (CD → loja).
- `contagem` (2.945): um item contado fisicamente numa unidade, em set/26.

**Colunas**

| Fonte | Coluna | O que é | Problemas vistos |
|---|---|---|---|
| estoque | `qtd` | saldo atual | 1.156 negativos (até −133); 1.118 zerados |
| estoque | `loja_id` | unidade (1–4 lojas, 5 site, 6 app, 7 CD) | site e app têm "estoque próprio", mas pela regra do Diego vendem do CD |
| estoque | `atualizado_em` | última mudança | de 01/01/2025 a 01/10/2026 (UTC) |
| entradas | `qtd` | quantidade recebida | no CD, 5.522 linhas sem quantidade (saídas anotadas como texto: "1 un -> savassi") |
| entradas | `obs` | tipo da entrada, em texto | "transf CD", "veio do CD", "transferencia" (~2.700 cada), "enviado pinheiros - 1 pçs"... |
| entradas | `fornecedor` | quem mandou | "CD" (5.476) e nomes variados para o mesmo fornecedor (Serra Azul ×4, Rio Claro ×3) |
| entradas | `custo_unit` | custo da peça | vazio em 16.446 (95%) |
| contagem | `nivel` | se contou por SKU ou por modelo | Savassi contou **por modelo** (161 linhas), sem cor nem tamanho |
| contagem | `data_contagem` | dia da contagem | só Pinheiros (20/09) e CD (28/09) têm data; Eldorado, Savassi e Batel não |
| contagem | `reservado` | peças separadas para pedidos | só no CD |
| contagem | `observacao` | recado | nome do conferente (Eldorado), "faltou contar provador" (46), "recontar" (36) |

**Problemas do estoque (lista única, nada decidido)**

| # | Problema | Quantos | Afeta |
|---|---|---|---|
| 1 | Saldo atual sem histórico (o ajuste sobrescreve) | toda a tabela | "quanto tinha há duas semanas" |
| 2 | A mesma peça com 2 códigos na mesma loja (longo positivo + curto negativo) | 741 | saldo por peça; explica parte dos negativos |
| 3 | Site e app com estoque espelhado (a mesma peça e quantidade nas duas "lojas") | 321 de 611 | estoque contado em dobro |
| 4 | CD: sistema 28.136 × WMS 6.601 (as saídas do CD não são descontadas) | CD inteiro | estoque do CD |
| 5 | Lojas: a contagem física dá mais que o sistema (Pinheiros 2.436 × 1.473; Eldorado 2.323 × 1.278; Savassi 2.114 × 1.497; Batel 1.881 × 1.224) | 4 lojas | qual fonte vale |
| 6 | Transferência anotada pelo CD, pela loja ou pelos dois | ~5.400 | risco de contar em dobro ou faltar |
| 7 | Contagens em datas diferentes (e 3 sem data); nenhuma em 01/10 | 5 unidades | chegar à posição de 01/10 |
| 8 | Savassi contada por modelo | 161 | não dá saldo por SKU |
| 9 | Custo vazio em 95% das entradas | 16.446 | custo do estoque |

**Ligações**
- SKU do estoque no catálogo: 47% escrito igual, 74% pela chave (os mesmos formatos do catálogo).
- Códigos da contagem (por SKU) no estoque: 99% pela chave. A contagem e o sistema falam das mesmas peças; o que diverge é a quantidade.

**Teste de reconstrução**
- **Fato:** entradas − vendas desde jan/25, comparado com a contagem física, dá estoque **negativo** (Pinheiros −1.209 × 2.428 contadas; Eldorado −683 × 2.310; Batel 82 × 1.881), e só 9–20% das peças batem. Motivos visíveis: as vendas começam em 01/01/2025, mas as primeiras entradas só semanas depois (não existe saldo inicial), e faltam movimentações (transferências e ajustes).
- **Em aberto (etapa 4):** como reconstruir o histórico com rastreabilidade desde o início. Nada decidido.

**Desafios da revisão**
- **O espelho site/app é o estoque do CD copiado:** em 230 das 232 peças espelhadas que estão no WMS, a quantidade do site e do app é **igual ao saldo do WMS do CD**. E 1.710 das 1.998 linhas do site/app foram atualizadas no mesmo dia, 29/09/2026 (um dia depois da posição do WMS, de 28/09). Tudo indica uma carga do WMS do CD gravada nas duas "lojas": somar site + app + CD conta o estoque do CD até três vezes.
- **A contagem não dá mais em todas as peças:** dá mais na maioria das divergências, mas há muitas iguais e muitas menores.

| Unidade | contado > sistema | igual | contado < sistema | só no sistema |
|---|---|---|---|---|
| Pinheiros | 385 | 471 | 100 | 38 |
| Eldorado | 607 | 281 | 297 | 263 |
| Batel | 390 | 451 | 87 | 37 |

- Eldorado tem 263 peças que existem no sistema e não aparecem na contagem: contagem incompleta? Vale olhar antes de confiar nela.

*(continuar preenchendo ao longo da análise)*